In [1]:
import torch
import os
from PIL import Image
from torch.utils.data import Dataset
from torchvision import transforms
from tqdm import tqdm
import random

device = torch.device("cuda:3" if torch.cuda.is_available() else "cpu")

num_epochs = 30
k_folds = 4
img_dir = 'kill_bill_time:400_80_20'
workspace = '3'
project_name='neuro-400-1'
img_size = 600
n_trials = 200
filename = '6000'

random_state = 42

In [2]:
class CirclesSequenceDataset(Dataset):
    def __init__(self, img_dir, img_size, transform=None, start_image = None, num_images=None, discret=10, sequence_length=1, prediction_length=3):
        self.img_dir = img_dir
        self.transform = transform or transforms.ToTensor()
        self.img_size = img_size
        self.discret = discret
        self.sequence_length = sequence_length
        self.prediction_length = prediction_length

        all_imgs = sorted(os.listdir(img_dir), key=lambda x: int(x.split('_')[0]))
        
        if num_images and start_image:
            self.imgs = all_imgs[start_image:start_image+num_images:self.discret]
        else:
            self.imgs = all_imgs[6::self.discret]

        if len(self.imgs) < self.sequence_length + self.prediction_length:
            raise ValueError(f"Недостаточно изображений для создания последовательностей (минимум {self.sequence_length + self.prediction_length} изображений).")

        self.cached_images = [None] * len(self.imgs)

        print("Кэширование изображений:")
        for idx in tqdm(range(len(self.imgs)), desc="Прогресс кэширования"):
            self._load_image(idx)

    def _load_image(self, idx):
        if self.cached_images[idx] is None:
            img_path = os.path.join(self.img_dir, self.imgs[idx])
            image = Image.open(img_path).convert("L")
            self.cached_images[idx] = self.transform(image)
        return self.cached_images[idx]

    def __len__(self):
        return len(self.imgs) - self.sequence_length - self.prediction_length + 1

    def __getitem__(self, idx):
        if idx + self.sequence_length + self.prediction_length > len(self.imgs):
            raise IndexError("Индекс выходит за пределы списка изображений.")

        # Получаем последовательность изображений
        imgs_sequence = [self._load_image(idx + i) for i in range(self.sequence_length)]
        imgs_sequence = torch.stack(imgs_sequence)

        # Собираем координаты меток в один тензор
        coords = []
        for i in range(self.prediction_length):
            label_img_name = self.imgs[idx + self.sequence_length + i]
            coord = label_img_name.split('.')[0].split('_')[1:]
            coords.extend([float(c) / self.img_size for c in coord])

        coords_tensor = torch.tensor(coords, dtype=torch.float32)

        return imgs_sequence, coords_tensor

In [3]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5]),  
])

In [4]:
from torch.utils.data import DataLoader, random_split

def get_data_loaders(dataset, batch_size, train_size=0.7):
    train_len = int(len(dataset) * train_size)
    val_len = len(dataset) - train_len

    # Разделение на тренировочную и валидационную выборки
    train_dataset, val_dataset = random_split(dataset, [train_len, val_len])

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, 
                          num_workers=20, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=True, 
                          num_workers=20, pin_memory=True)

    print(f"Размер тренировочной выборки: {len(train_dataset)}")
    print(f"Размер валидационной выборки: {len(val_dataset)}")
    
    return train_loader, val_loader

dataset_train = CirclesSequenceDataset(transform = transform, img_dir=img_dir, img_size=img_size, start_image = 6, num_images = 50000)
dataset_test = CirclesSequenceDataset(transform = transform, img_dir=img_dir, img_size=img_size, start_image = 61000, num_images = 10000)

Кэширование изображений:


Прогресс кэширования: 100%|█████████████████| 5000/5000 [15:17<00:00,  5.45it/s]


Кэширование изображений:


Прогресс кэширования: 100%|█████████████████| 1000/1000 [03:35<00:00,  4.64it/s]


In [5]:
import comet_ml
import torch.optim as optim
import torch.nn as nn 
from sklearn.metrics import r2_score, mean_absolute_error, mean_absolute_percentage_error
import optuna
from optuna import Trial
from thop import profile  # Для подсчета FLOPs
import numpy as np
from sklearn.model_selection import KFold
import gc
from sklearn.model_selection import ShuffleSplit
from torch.cuda.amp import autocast, GradScaler
import time
import torch
from sklearn.metrics import mean_squared_error

log_file_path = "600_normal_4.txt"


# Функция для вычисления ATE - среднее расстояние между реальными и предсказанными позициями
def calculate_ate(predictions, ground_truths):
    # Убедимся, что входные данные - это массивы NumPy
    predictions = np.array(predictions)*600
    ground_truths = np.array(ground_truths)*600
    
    return np.mean(np.linalg.norm(predictions - ground_truths, axis=1))
    
# Функция для вычисления RPE - оценивает отклонение между предсказанными и истинными относительными позами (например, между соседними точками траектории)
def calculate_rpe(predictions, ground_truths):
    # Преобразуем входные данные в NumPy массивы
    predictions = np.array(predictions)*600
    ground_truths = np.array(ground_truths)*600
    
    # Вычисляем изменения позы для предсказаний и истинных значений
    pred_deltas = np.diff(predictions, axis=0)
    gt_deltas = np.diff(ground_truths, axis=0)
    
    # Вычисляем отклонение между изменениями позы
    rpe = np.linalg.norm(pred_deltas - gt_deltas, axis=1)
    return np.mean(rpe)

def calculate_rmse(predictions, ground_truths):
    # Преобразуем входные данные в NumPy массивы
    predictions = np.array(predictions)*600
    ground_truths = np.array(ground_truths)*600
    
    # Вычисляем среднеквадратичную ошибку
    rmse = np.sqrt(np.mean((predictions - ground_truths) ** 2))
    
    return rmse

def calculate_fde(predicted, true):

    predicted = np.array(predicted)*600
    true = np.array(true)*600
    
    # Возвращаем расстояние между последними предсказанными и истинными координатами
    return np.linalg.norm(predicted[-1] - true[-1])

# Функция для освобождения памяти
def free_memory():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

# Функция сохранения в файл дополнительных логгов
def log_to_file(message):
    with open(log_file_path, "a") as log_file:
        log_file.write(message + "\n")

# Вычисление суммарного количества FLOPs на инференсе
def calculate_flops(model, input_size):
    inputs = torch.randn(input_size).to(device)
    flops, params = profile(model, inputs=(inputs,), verbose=False)
    return flops

In [6]:
#Общая модель
class EncDec(nn.Module):
    def __init__(self, filters_1, filters_2, hidden_size, num_layers, dropout_rate, kernel_size_1, kernel_size_2, 
                 stride_1, stride_2, padding_1, padding_2, use_second_conv):
        super(EncDec, self).__init__()
        
        self.Conv2d_1 = nn.Conv2d(in_channels=1, out_channels=filters_1, kernel_size=kernel_size_1, stride=stride_1, padding=padding_1)
        self.BatchNorm_1 = nn.BatchNorm2d(filters_1)
        self.MaxPool_1 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.High = (img_size - kernel_size_1 + 2 * padding_1) // stride_1 + 1
        self.High = (self.High - 2) // 2 + 1  # После MaxPool

        self.use_second_conv = use_second_conv
        if use_second_conv:
            self.Conv2d_2 = nn.Conv2d(in_channels=filters_1, out_channels=filters_2, kernel_size=kernel_size_2, stride=stride_2, padding=padding_2)
            self.BatchNorm_2 = nn.BatchNorm2d(filters_2)
            self.MaxPool_2 = nn.MaxPool2d(kernel_size=2, stride=2)

            self.High = (self.High - kernel_size_2 + 2 * padding_2) // stride_2 + 1
            self.High = (self.High - 2) // 2 + 1 

        conv_output_size = filters_2 if use_second_conv else filters_1
        self.LSTM = nn.LSTM(conv_output_size * self.High * self.High, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, 6)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        batch_size = x.size(0)
        seq_len = x.size(1)
        
        x = x.permute(0, 2, 1, 3, 4)
        
        conv2d_outputs = []
        for i in range(seq_len):

            conv2d_out = torch.relu(self.BatchNorm_1(self.Conv2d_1(x[:, :, i, :, :])))
            conv2d_out = self.MaxPool_1(conv2d_out)
        
            if self.use_second_conv:
                conv2d_out = torch.relu(self.BatchNorm_2(self.Conv2d_2(conv2d_out)))
                conv2d_out = self.MaxPool_2(conv2d_out)
        
            conv2d_outputs.append(conv2d_out.unsqueeze(1))
        
        conv_outputs = torch.cat(conv2d_outputs, dim=1)
        
        z = conv_outputs.reshape(batch_size, seq_len, -1)
        
        output, (hidden, cell) = self.LSTM(z)
        
        pred = self.fc(output[:, -1, :])
        
        return self.sigmoid(pred)


def train(model, train_loader, criterion, optimizer, device):
    
    model.train()  
    total_loss = 0.0
    predictions, ground_truths = [], []

    scaler = GradScaler()

    for imgs_sequence, coords_tensor in tqdm(train_loader, leave=False):
        imgs_sequence = imgs_sequence.to(device, non_blocking=True)
        coords_tensor = coords_tensor.to(device, non_blocking=True)

        optimizer.zero_grad()

        with autocast():
            outputs = model(imgs_sequence)

            if outputs.size() != coords_tensor.size():
                raise ValueError(f"Размеры outputs {outputs.size()} и coords {coords_tensor.size()} не совпадают.")

            loss = criterion(outputs, coords_tensor)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * imgs_sequence.size(0)
        predictions.extend(outputs.detach().cpu().numpy())
        ground_truths.extend(coords_tensor.detach().cpu().numpy())

    del imgs_sequence, coords_tensor, outputs, loss
    torch.cuda.empty_cache()
    gc.collect()

    return total_loss / len(train_loader.dataset), predictions, ground_truths


def evaluate(model, val_loader, criterion, device):
    
    model.eval() 
    total_loss = 0.0
    predictions, ground_truths = [], []

    with torch.no_grad():  
        for imgs_sequence, coords_tensor in tqdm(val_loader, leave=False):
            imgs_sequence = imgs_sequence.to(device, non_blocking=True)
            coords_tensor = coords_tensor.to(device, non_blocking=True)

            outputs = model(imgs_sequence)

            if outputs.size() != coords_tensor.size():
                raise ValueError(f"Размеры outputs {outputs.size()} и coords {coords_tensor.size()} не совпадают.")

            loss = criterion(outputs, coords_tensor)

            total_loss += loss.item() * imgs_sequence.size(0)
            predictions.extend(outputs.detach().cpu().numpy())
            ground_truths.extend(coords_tensor.detach().cpu().numpy())

        del imgs_sequence, coords_tensor, outputs, loss
        torch.cuda.empty_cache()
        gc.collect()

    return total_loss / len(val_loader.dataset), predictions, ground_truths

In [7]:
def objective(trial: optuna.Trial, n_splits=2, img_size=600):
    """K-Fold кросс-валидация с усреднением всех метрик по фолдам."""
    # Гиперпараметры для текущего trial
    use_second_conv = trial.suggest_categorical('use_second_conv', [True, False])
    batch_size = trial.suggest_categorical('batch_size', [4, 6, 8, 10, 16, 32])
    filters_1 = trial.suggest_int('filters_1', 2, 16)
    filters_2 = trial.suggest_int('filters_2', 2, 32) if use_second_conv else filters_1
    hidden_size = trial.suggest_int('hidden_size', 4, 256)
    num_layers = trial.suggest_int('num_layers', 1, 3)
    dropout_rate = trial.suggest_float('dropout_rate', 0.1, 0.5)
    kernel_size_1 = trial.suggest_int('kernel_size_1', 3, 5)
    kernel_size_2 = trial.suggest_int('kernel_size_2', 3, 5)
    stride_1 = trial.suggest_int('stride_1', 1, 2)
    stride_2 = trial.suggest_int('stride_2', 1, 2)
    padding_1 = trial.suggest_int('padding_1', 1, 2)
    padding_2 = trial.suggest_int('padding_2', 1, 2)
    learning_rate = trial.suggest_float('learning_rate', 1e-5, 1e-2, log=True)
    
    model = EncDec(
    filters_1=filters_1, filters_2=filters_2, hidden_size=hidden_size,
    num_layers=num_layers, dropout_rate=dropout_rate,
    kernel_size_1=kernel_size_1, kernel_size_2=kernel_size_2,
    stride_1=stride_1, stride_2=stride_2,
    padding_1=padding_1, padding_2=padding_2,
    use_second_conv=use_second_conv
    ).to(device)
    
    # Создание DataLoader'ов для тренировочного и валидационного наборов
    train_loader = DataLoader(dataset_train, batch_size=batch_size, num_workers=20, pin_memory=True, persistent_workers=True, shuffle=True)
    val_loader = DataLoader(dataset_test, batch_size=batch_size, num_workers=20, pin_memory=True, persistent_workers=True, shuffle=False)

    optimizer = optim.AdamW(model.parameters(), lr=learning_rate)
    criterion = nn.MSELoss()

    train_losses = []
    val_losses = []
    epochs = [] 


    for epoch in range(num_epochs):
        try:

            train_loss, train_preds, train_gt = train(model, train_loader, criterion, optimizer, device)
            # val_loss, val_preds, val_gt = evaluate(model, val_loader, criterion, device)


            # train_losses.append(train_loss)
            # val_losses.append(val_loss)
            # epochs.append(epoch)
            

            if epoch == num_epochs - 1:
                val_loss, val_preds, val_gt = evaluate(model, val_loader, criterion, device)
                rmse_val = calculate_rmse(val_preds, val_gt)
                ate_val = calculate_ate(val_preds, val_gt)
                rpe_val = calculate_rpe(val_preds, val_gt)
                fde_val = calculate_fde(val_preds, val_gt)
                
                del val_preds, val_gt

            free_memory()
            del train_preds, train_gt
            gc.collect()
            torch.cuda.empty_cache()

        except torch.cuda.OutOfMemoryError as e:
            print(f"Ошибка переполнения памяти: {str(e)}")
            torch.cuda.empty_cache()
            gc.collect()
            raise optuna.exceptions.TrialPruned()

    gc.collect()
    torch.cuda.empty_cache()
    free_memory()

    experiment = comet_ml.Experiment(api_key="VHqjhRzLpPJbb986xCh3V3ei2", project_name=project_name, workspace=workspace)

    experiment.log_metric("train_loss_graph", train_loss)
    experiment.log_metric("val_loss_graph", val_loss)

    experiment.log_parameters(trial.params)
    experiment.log_metric("rmse", rmse_val)
    experiment.log_metric("ate", ate_val)
    experiment.log_metric("rpe", rpe_val)
    experiment.log_metric("fde", fde_val)
    
    flops = calculate_flops(model, (batch_size, 1, 1, 600, 600))
    experiment.log_metric("flops", flops)
    experiment.end()

    return rmse_val

In [ ]:
study = optuna.create_study(direction='minimize', pruner=optuna.pruners.NopPruner())
study.optimize(objective, n_trials=n_trials)
torch.cuda.set_per_process_memory_fraction(0.98) 

best_trials = study.best_trial

# with open(f'{filename}', 'a') as f:
#     for trial in best_trials:
#         f.write(f"Лучшие параметры для trial {trial.number}:\n")
#         for key, value in trial.params.items():
#             f.write(f"{key}: {value}\n")
    
#         f.write(f"\nЛучшие метрики для trial {trial.number}:\n")
#         f.write(f"R² на валидации: {trial.values[0]}:\n")

[I 2024-12-10 18:49:57,639] A new study created in memory with name: no-name-47699249-0f68-4a12-917d-902c415a7faa
/tmp/ipykernel_1994911/2516994783.py:63: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
  0%|                                                   | 0/833 [00:00<?, ?it/s]/tmp/ipykernel_1994911/2516994783.py:71: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
COMET WARNING: To get all data logged automatically, import comet_ml before the following modules: torch.
COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.
COMET INFO: Experiment is live on comet.com https://www.comet.com/3/neuro-400-1/77f3471557fd43d68428f75954b225fd

COMET INFO: Couldn't find a Git reposi